Дообучение сохранённых моделей до 100% validation accuracy на всех маленьких буквах
и больших классах-донорах. Большая H остаётся только тестом.
Исходные результаты: `letters_scale_epochwise`; продолжение: `letters_scale_val100`.
Повторный запуск продолжает незавершённые условия и пропускает уже достигшие цели.

In [1]:
from pathlib import Path
import os
import csv
import sys
import time
import hashlib

import numpy as np
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
from torch.nn import functional as F

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "letters.pt"
SOURCE_RESULTS = ROOT / "results" / "letters_scale_epochwise"
RESULTS = ROOT / "results" / "letters_scale_val100"
SEED = 42
TARGET = "H"
TRAIN_PER_CLASS = 128
BATCH_SIZE = 64
MAX_EXTRA_EPOCHS = 200
TARGET_ACCURACY = 1.0
LR = 2e-5
RESUME_LR_CAP = 5e-6
WEIGHT_DECAY = 0.01
DONOR_COUNTS = (0, 1, 5, 10, 15, 20, 25)

assert torch.cuda.is_available(), "Нужна CUDA-сборка PyTorch."
DEVICE = torch.device("cuda")
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from model import DeiTTiny
print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
data = torch.load(DATA_PATH, map_location="cpu", weights_only=True)
images = data["images"]
LETTERS = data["letters"]
HEIGHTS = data["heights"]
CANVAS = data["canvas"]
pools = data["shift_pools"]
TEST_POSITIONS = len(pools[0]["test"])
assert all(len(pool["test"]) == TEST_POSITIONS for pool in pools)
print("Loaded:", DATA_PATH, tuple(images.shape))


Loaded: C:\Users\Mikhail\Desktop\projects\symmetry_priors\data\letters.pt (26, 2, 1, 224, 224)


Сохраняем исходный состав доноров, распределение размеров и train/validation/test.
По 128 предъявлений на класс за эпоху; у доноров половина большая. Сдвиги случайные,
независимые для каждого изображения, только из train-координат. H большой в обучении нет.
Модель стартует с ранее обученных весов. Старые checkpoint не содержали AdamW: при первом
дообучении его состояние создаётся заново. Затем сохраняются optimizer и состояния генераторов.

In [3]:
templates = images.to(DEVICE, dtype=torch.float32).div(255)


def make_batch(labels, sizes, shifts):
    labels, sizes, shifts = labels.to(DEVICE), sizes.to(DEVICE), shifts.to(DEVICE)
    source = templates[labels, sizes]
    theta = torch.eye(2, 3, device=DEVICE).unsqueeze(0).repeat(len(labels), 1, 1)
    theta[:, 0, 2] = -2 * shifts[:, 0] / CANVAS
    theta[:, 1, 2] = -2 * shifts[:, 1] / CANVAS
    grid = F.affine_grid(theta, source.shape, align_corners=False)
    shifted = F.grid_sample(source, grid, mode="nearest", padding_mode="zeros", align_corners=False)
    return shifted.expand(-1, 3, -1, -1)


def classification_metrics(logits, labels):
    log_probabilities = logits.log_softmax(-1)
    probabilities = log_probabilities.exp()
    predicted = logits.argmax(-1)
    correct = predicted.eq(labels)
    true_probability = probabilities.gather(1, labels[:, None]).squeeze(1)
    return {
        "loss": F.cross_entropy(logits, labels).item(),
        "accuracy": correct.float().mean().item(),
        "true_confidence": true_probability.mean().item(),
        "entropy": -(probabilities * log_probabilities).sum(-1).mean().item(),
        "confidence_correct": true_probability[correct].mean().item() if correct.any() else float("nan"),
        "correct": correct.sum().item(), "count": len(labels),
        "predicted_counts": torch.bincount(predicted, minlength=len(LETTERS)).tolist(),
        "probabilities": probabilities,
    }


@torch.inference_mode()
def evaluate(model, classes, size_index, split):
    model.eval()
    coordinates = pools[size_index][split]
    labels = torch.tensor(classes).repeat_interleave(len(coordinates))
    shifts = coordinates.repeat(len(classes), 1)
    logits = []
    for start in range(0, len(labels), BATCH_SIZE):
        batch_labels = labels[start:start + BATCH_SIZE]
        sizes = torch.full_like(batch_labels, size_index)
        x = make_batch(batch_labels, sizes, shifts[start:start + BATCH_SIZE])
        logits.append(model(x).cpu())
    return classification_metrics(torch.cat(logits), labels)


def upsert_csv(path, row, keys):
    rows = []
    if path.exists():
        with path.open(newline="", encoding="utf-8") as file:
            rows = list(csv.DictReader(file))
    rows = [old for old in rows if any(str(old[key]) != str(row[key]) for key in keys)]
    temporary = path.with_suffix(".csv.tmp")
    with temporary.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(row))
        writer.writeheader()
        writer.writerows(rows + [row])
    temporary.replace(path)


base_labels = torch.arange(len(LETTERS)).repeat_interleave(TRAIN_PER_CLASS)
donor_order = LETTERS.replace(TARGET, "")
all_classes = list(range(len(LETTERS)))

Каждую эпоху сохраняются все тестовые вероятности, train/validation/test метрики и веса.
NPZ включает исходную историю и новые эпохи, без повторов номеров. CSV содержит аудит исходного
checkpoint и историю дообучения. Веса сохраняются вместе с optimizer и генераторами для продолжения.
Остановка: ноль ошибок на validation маленьких букв и больших доноров; контрольная большая H
не влияет на остановку, learning rate и обновление весов. LR уменьшается вдвое после 8 эпох
без улучшения validation loss. Ограничение 200 дополнительных эпох за один запуск — страховка;
если цель не достигнута, повторный запуск продолжит сохранённое состояние.
100% на этой конечной валидации не означает 100% на тесте или всех возможных изображениях.
При продолжении незавершённого условия LR ограничивается RESUME_LR_CAP = 5e-6;
состояние AdamW сохраняется. В уже выполненном первом проходе использовался LR = 2e-5.
CSV обновляются атомарно, последняя строка истории также сохраняется в checkpoint.


In [4]:
RESULTS.mkdir(parents=True, exist_ok=True)
metrics_path = RESULTS / "metrics.csv"
history_path = RESULTS / "history.csv"
results = []

In [5]:
for donor_count in DONOR_COUNTS:
    run_name = f"big_{donor_count:02d}"
    source_path = SOURCE_RESULTS / f"{run_name}.pt"
    checkpoint_path = RESULTS / f"{run_name}.pt"
    resumed = checkpoint_path.exists()
    checkpoint = torch.load(checkpoint_path if resumed else source_path,
                            map_location="cpu", weights_only=True)
    donors = checkpoint["donors"]
    assert donors == donor_order[:donor_count] and TARGET not in donors
    donor_classes = [LETTERS.index(letter) for letter in donors]
    augmented = torch.tensor([letter in donors for letter in LETTERS])
    epoch = int(checkpoint["epoch"])
    source_epoch = int(checkpoint.get("source_epoch", epoch))
    source_sha = checkpoint.get("source_sha256") or hashlib.sha256(source_path.read_bytes()).hexdigest()

    torch.manual_seed(SEED)
    model = DeiTTiny(num_classes=len(LETTERS)).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, fused=True)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=8, min_lr=1e-6)
    generators = {"order": torch.Generator().manual_seed(744),
                  "shift": torch.Generator().manual_seed(745),
                  "size": torch.Generator().manual_seed(746)}
    if resumed:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if not checkpoint.get("val_target_met", False):
            for group in optimizer.param_groups:
                group["lr"] = min(group["lr"], RESUME_LR_CAP)
        for name, generator in generators.items():
            generator.set_state(checkpoint["rng_states"][name])
    previous_history = checkpoint.get("history_row")
    if resumed and previous_history is None and history_path.exists():
        with history_path.open(newline="", encoding="utf-8") as file:
            previous_history = next((r for r in csv.DictReader(file)
                                     if r["run"] == run_name and int(r["epoch"]) == epoch), None)
    if previous_history is not None:
        upsert_csv(history_path, previous_history, ("run", "epoch"))
    del checkpoint

    probability_path = RESULTS / f"{run_name}_probabilities.npz"
    archive_path = probability_path if resumed else SOURCE_RESULTS / probability_path.name
    with np.load(archive_path, allow_pickle=False) as saved:
        probability_epochs = saved["epochs"].tolist()
        keep = [i for i, value in enumerate(probability_epochs) if value <= epoch]
        probability_epochs = [probability_epochs[i] for i in keep]
        probability_history = [p.copy() for p in saved["probabilities"][keep]]
        metadata = {key: saved[key].copy() for key in saved.files
                    if key not in ("probabilities", "epochs")}
    assert probability_epochs[-1] == epoch
    metadata["source_epoch"] = np.array(source_epoch)
    metadata["source_sha256"] = np.array(source_sha)
    started = time.perf_counter()
    print(f"{run_name}: checkpoint epoch={epoch}, donors={donors or '-'}", flush=True)

    for extra_epoch in range(MAX_EXTRA_EPOCHS + 1):
        train_loss = train_accuracy = float("nan")
        used_lr = optimizer.param_groups[0]["lr"]
        if extra_epoch:
            epoch += 1
            order = torch.randperm(len(base_labels), generator=generators["order"])
            size_flags = torch.zeros(len(LETTERS), TRAIN_PER_CLASS, dtype=torch.bool)
            for c in all_classes:
                indices = torch.randperm(TRAIN_PER_CLASS, generator=generators["size"])
                size_flags[c, indices[:TRAIN_PER_CLASS // 2]] = True
            size_flags = size_flags.flatten()
            model.train()
            total_loss, total_correct = 0.0, 0
            for ids in order.split(BATCH_SIZE):
                labels = base_labels[ids]
                sizes = (augmented[labels] & size_flags[ids]).long()
                choices = [pool["train"][torch.randint(len(pool["train"]), (len(ids),),
                            generator=generators["shift"])] for pool in pools]
                shifts = torch.where(sizes[:, None].bool(), choices[1], choices[0])
                x = make_batch(labels, sizes, shifts)
                labels = labels.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(ids)
                total_correct += logits.argmax(-1).eq(labels).sum().item()
            train_loss = total_loss / len(base_labels)
            train_accuracy = total_correct / len(base_labels)

        small_val = evaluate(model, all_classes, 0, "val")
        large_val = evaluate(model, donor_classes, 1, "val") if donor_classes else None
        reached = (small_val["correct"] == small_val["count"] and
                   (large_val is None or large_val["correct"] == large_val["count"]))
        if extra_epoch:
            val_loss = small_val["loss"] * small_val["count"]
            val_count = small_val["count"]
            if large_val:
                val_loss += large_val["loss"] * large_val["count"]
                val_count += large_val["count"]
            scheduler.step(val_loss / val_count)
        small = evaluate(model, all_classes, 0, "test")
        large = evaluate(model, [LETTERS.index(TARGET)], 1, "test")
        probabilities = torch.cat([small["probabilities"], large["probabilities"]]).numpy()
        if probability_epochs[-1] == epoch:
            probability_history[-1] = probabilities
        else:
            probability_epochs.append(epoch)
            probability_history.append(probabilities)

        probability_temp = probability_path.with_suffix(".npz.tmp")
        with probability_temp.open("wb") as file:
            np.savez(file, probabilities=np.stack(probability_history),
                     epochs=np.array(probability_epochs), **metadata)
        probability_temp.replace(probability_path)
        history_row = {"run": run_name, "epoch": epoch, "stage": "finetune" if extra_epoch else "audit",
                       "lr": used_lr, "train_loss": train_loss, "train_accuracy": train_accuracy,
                       "val_small_loss": small_val["loss"], "val_small_accuracy": small_val["accuracy"],
                       "val_small_correct": small_val["correct"], "val_small_count": small_val["count"],
                       "val_donor_large_accuracy": large_val["accuracy"] if large_val else float("nan"),
                       "val_donor_large_loss": large_val["loss"] if large_val else float("nan"),
                       "val_donor_large_correct": large_val["correct"] if large_val else 0,
                       "val_donor_large_count": large_val["count"] if large_val else 0,
                       "test_small_loss": small["loss"], "test_small_accuracy": small["accuracy"],
                       "test_small_true_confidence": small["true_confidence"],
                       "test_h_loss": large["loss"], "test_h_recall": large["accuracy"],
                       "test_h_true_confidence": large["true_confidence"],
                       "test_h_entropy": large["entropy"], "val_target_met": reached}
        if not extra_epoch and previous_history is not None:
            history_row = previous_history
        state = {"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                 "letters": LETTERS, "donors": donors, "seed": SEED, "epoch": epoch,
                 "source_epoch": source_epoch, "source_sha256": source_sha,
                 "history_row": history_row,
                 "val_target_met": reached, "optimizer_restarted_at_source": True,
                 "optimizer_state_dict": optimizer.state_dict(),
                 "scheduler_state_dict": scheduler.state_dict(),
                 "rng_states": {name: generator.get_state() for name, generator in generators.items()}}
        checkpoint_temp = checkpoint_path.with_suffix(".pt.tmp")
        torch.save(state, checkpoint_temp)
        checkpoint_temp.replace(checkpoint_path)
        del state

        if extra_epoch or not resumed:
            upsert_csv(history_path, history_row, ("run", "epoch"))
        row = {"run": run_name, "donors": donors, "donor_count": donor_count,
               "source_epoch": source_epoch, "epochs": epoch, "extra_epochs": epoch - source_epoch,
               "val_small_accuracy": small_val["accuracy"],
               "val_small_correct": small_val["correct"], "val_small_count": small_val["count"],
               "val_donor_large_accuracy": large_val["accuracy"] if large_val else float("nan"),
               "val_donor_large_correct": large_val["correct"] if large_val else 0,
               "val_donor_large_count": large_val["count"] if large_val else 0,
               "val_target_met": reached, "small_accuracy": small["accuracy"],
               "small_correct": small["correct"], "small_count": small["count"],
               "large_recall": large["accuracy"], "large_correct": large["correct"],
               "large_count": large["count"], "small_true_confidence": small["true_confidence"],
               "large_true_confidence": large["true_confidence"],
               "small_loss": small["loss"], "large_loss": large["loss"],
               "small_entropy": small["entropy"], "large_entropy": large["entropy"]}
        upsert_csv(metrics_path, row, ("run",))
        seen = f"{large_val['correct']}/{large_val['count']}" if large_val else "n/a"
        print(f"  epoch {epoch}: val small={small_val['correct']}/{small_val['count']}, "
              f"val large={seen}, H recall={large['correct']}/{large['count']}, "
              f"p(H)={large['true_confidence']:.6g}, {time.perf_counter()-started:.0f}s", flush=True)
        if reached:
            break
    results.append(row)
    del model, optimizer, scheduler, probability_history
    torch.cuda.empty_cache()
    if not reached:
        print(f"{run_name}: лимит дополнительных эпох; цель ещё не достигнута.", flush=True)

big_00: checkpoint epoch=91, donors=-
  epoch 91: val small=3328/3328, val large=n/a, H recall=0/256, p(H)=4.15473e-05, 11s
big_01: checkpoint epoch=100, donors=A
  epoch 100: val small=3328/3328, val large=128/128, H recall=0/256, p(H)=4.4549e-07, 11s
big_05: checkpoint epoch=172, donors=ABCDE
  epoch 172: val small=3328/3328, val large=640/640, H recall=0/256, p(H)=0.000293742, 12s
big_10: checkpoint epoch=90, donors=ABCDEFGIJK
  epoch 90: val small=3328/3328, val large=1280/1280, H recall=0/256, p(H)=0.000580466, 12s
big_15: checkpoint epoch=171, donors=ABCDEFGIJKLMNOP
  epoch 171: val small=3324/3328, val large=1919/1920, H recall=0/256, p(H)=2.17211e-05, 13s
  epoch 172: val small=3323/3328, val large=1920/1920, H recall=0/256, p(H)=2.54174e-05, 38s
  epoch 173: val small=3325/3328, val large=1920/1920, H recall=0/256, p(H)=3.05494e-05, 63s
  epoch 174: val small=3324/3328, val large=1920/1920, H recall=0/256, p(H)=2.80688e-05, 87s
  epoch 175: val small=3325/3328, val large=1920/

In [6]:
print(f"{'Run':10s} {'Epochs':>7s} {'Added':>6s} {'Val 100%':>9s} {'Test small':>11s} {'H recall':>10s}")
for row in results:
    print(f"{row['run']:10s} {row['epochs']:7d} {row['extra_epochs']:6d} "
          f"{str(row['val_target_met']):>9s} {row['small_accuracy']:11.4f} {row['large_recall']:10.4f}")
print("Saved:", RESULTS)

Run         Epochs  Added  Val 100%  Test small   H recall
big_00          91      1      True      1.0000     0.0000
big_01         100      0      True      1.0000     0.0000
big_05         172     52      True      1.0000     0.0000
big_10          90      0      True      1.0000     0.0000
big_15         185     65      True      0.9998     0.0000
big_20          95      0      True      1.0000     0.0000
big_25         100      0      True      0.9997     0.0000
Saved: C:\Users\Mikhail\Desktop\projects\symmetry_priors\results\letters_scale_val100
